In [ ]:
# ==============================================================================
# 01_utility_experiments.ipynb
# Purpose: Benchmark Utility-Privacy Tradeoff across varying noise levels (epsilon)
# ==============================================================================

import os
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Updated imports matching your module naming
from src.dataset_loading import prepare_data
from src.models import DPMLPClassifier
from src.loss_def import BinaryFocalLoss
from src.dp_engine import run_dp_experiment

# Set figure styles for report inclusion (clean, readable fonts)
plt.rcParams.update({'font.size': 12, 'figure.autolayout': True})
os.makedirs("../results/figures", exist_ok=True)

# 1. Hyperparameters & Configuration
DATA_PATH = "../data/creditcard.csv"
BATCH_SIZE = 512
EPOCHS = 8
MAX_GRAD_NORM = 1.0
TARGET_DELTA = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Noise multipliers to sweep over (maps to varied final epsilons)
NOISE_MULTIPLIERS = [4.0, 2.0, 1.2, 0.8, 0.5]

# 2. Data Preparation
train_loader, test_loader, pos_weight, input_dim = prepare_data(
    DATA_PATH, batch_size=BATCH_SIZE
)

# 3. Execution Loop
results = []

for sigma in NOISE_MULTIPLIERS:
    print(f"\n--- Running Experiment with Noise Multiplier σ = {sigma} ---")
    
    # Re-initialize fresh model & optimizer
    model = DPMLPClassifier(input_dim=input_dim, hidden_dim=64)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = BinaryFocalLoss(gamma=2.0, pos_weight=pos_weight)

    model, history = run_dp_experiment(
        train_loader=train_loader,
        test_loader=test_loader,
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        epochs=EPOCHS,
        max_grad_norm=MAX_GRAD_NORM,
        noise_multiplier=sigma,
        target_delta=TARGET_DELTA,
        device=DEVICE
    )
    
    final_eps = history["epsilon"][-1]
    final_roc = history["roc_auc"][-1]
    final_pr = history["pr_auc"][-1]
    
    results.append({
        "sigma": sigma,
        "epsilon": final_eps,
        "roc_auc": final_roc,
        "pr_auc": final_pr,
        "history": history
    })

# Convert summary results to DataFrame
df_res = pd.DataFrame(results)
print("\n=== FINAL EXPERIMENT SUMMARY ===")
print(df_res[["sigma", "epsilon", "roc_auc", "pr_auc"]])

# 4. Plotting: Utility vs. Privacy Budget (ε)
fig, ax1 = plt.subplots(figsize=(7, 4.5))

color_roc = 'tab:blue'
ax1.set_xlabel('Privacy Parameter ε (Lower = Higher Privacy)', fontweight='bold')
ax1.set_ylabel('ROC-AUC', color=color_roc, fontweight='bold')
ax1.plot(df_res['epsilon'], df_res['roc_auc'], marker='o', color=color_roc, linewidth=2, label='ROC-AUC')
ax1.tick_params(axis='y', labelcolor=color_roc)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()  
color_pr = 'tab:red'
ax2.set_ylabel('PR-AUC (Fraud Class Focus)', color=color_pr, fontweight='bold')
ax2.plot(df_res['epsilon'], df_res['pr_auc'], marker='s', color=color_pr, linewidth=2, linestyle='--', label='PR-AUC')
ax2.tick_params(axis='y', labelcolor=color_pr)

plt.title('DP-SGD Utility-Privacy Trade-off (Credit Card Fraud)', fontsize=13)
plt.savefig("../results/figures/utility_vs_epsilon.pdf", bbox_inches='tight')
plt.savefig("../results/figures/utility_vs_epsilon.png", dpi=300, bbox_inches='tight')
plt.show()